# Task 2: End-to-End ML Pipeline with Scikit-learn Pipeline API
### **Internship:** DeveloperHub Corporation — AI/ML Engineering

---

## 📌 Problem Statement & Objective
Predict customer churn (whether a customer will leave the telecom company or stay) by building a reusable, production-ready, and optimized machine learning pipeline.

The dataset contains structured customer attributes (tenure, contract type, payment method, monthly charges, etc.). This is a **Binary Classification** problem where we handle different data types (numerical and categorical) simultaneously, tune hyperparameters using cross-validation, and export the complete pipeline for immediate deployment.

---

## ⚙️ Preprocessing & Engineering Workflow
* **Data Cleaning:** Handled missing values/empty spaces in `TotalCharges` dynamically using median imputation.
* **Feature Scaling:** Applied `StandardScaler` on numerical columns ($X$ variables like tenure and charges) to normalize the scale.
* **Categorical Encoding:** Transformed textual categories (like contract type and internet service) into numeric matrices using `OneHotEncoder`.
* **Feature Fusion:** Integrated both preprocessing tracks into a single production-ready unified `ColumnTransformer`.

---

## 🤖 Machine Learning Models & Optimization
* **Base Classifiers:** Evaluated robust algorithms including **Logistic Regression** and **Random Forest Classifier**.
* **Hyperparameter Tuning:** Implemented **GridSearchCV** with 3-fold cross-validation to search the optimal combination of tree depth (`max_depth`) and estimators (`n_estimators`).
* **Artifact Export:** Saved the entire pipeline (Preprocessing + Best Model) into a single reusable `.pkl` file using **Joblib** for strict zero-leakage production readiness.

---

## 📊 Evaluation Metrics
* **Accuracy:** To measure the overall percentage of correct predictions (Stay vs. Leave).
* **F1-Score (Macro/Weighted):** The critical metric for this task due to class imbalance, balancing both Precision and Recall to ensure the model doesn't ignore the minority "Churn" class.

# Step 1: Environment Setup & Library Installation
Is cell mein hum un tamam libraries ko install aur import kar rahe hain jo hamare data ko preprocess karne, machine learning model train karne aur aakhiri step mein Gradio UI (web app) banane ke liye zaroori hain.

In [ ]:
!pip install -U pandas numpy scikit-learn gradio joblib requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 70.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.3/73.3 kB 2.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: starlette
    Found existing installa

# Step 2: Data Acquisition & Pre-cleaning
Is step mein hum IBM ke public server se raw Telco Churn dataset download kar rahe hain. Data load karne ke baad hum teen zaroori kaam kar rahe hain:
1. `customerID` ko delete kar rahe hain kyunki ID ka customer ke chornay ya ruknay se koi talluq nahi hota.
2. `TotalCharges` column mein jo khali jaghein (empty spaces) thin, unhein missing values (`NaN`) mein badal kar column ko numeric data type de rahe hain.
3. Missing values ko pure column ke `median` value se fill kar rahe hain aur target column `Churn` ko 'Yes/No' se `1/0` (binary format) mein badal rahe hain.

In [ ]:
import pandas as pd
import numpy as np

print("Loading Telco Churn Dataset...")
# Public reliable source to fetch the raw Telco Churn CSV
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

# 1. Drop CustomerID (Yeh prediction mein kaam nahi aata)
df.drop(columns=["customerID"], inplace=True)

# 2. TotalCharges column mein empty spaces ko NaN mein convert karein aur float banayein
df['TotalCharges'] = df['TotalCharges'].replace(r'^\s*$', np.nan, regex=True)
df['TotalCharges'] = df['TotalCharges'].astype(float)

# 3. Missing values ko fill karein (Median se)
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

# 4. Target variable 'Churn' ko binary (0 aur 1) mein convert karein
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

print("Dataset Loaded and Cleaned Successfully!")
print(f"Dataset Shape: {df.shape}")
print(df.head(2))

Loading Telco Churn Dataset...
Dataset Loaded and Cleaned Successfully!
Dataset Shape: (7043, 20)
   gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  Female              0     Yes         No       1           No   
1    Male              0      No         No      34          Yes   

      MultipleLines InternetService OnlineSecurity OnlineBackup  \
0  No phone service             DSL             No          Yes   
1                No             DSL            Yes           No   

  DeviceProtection TechSupport StreamingTV StreamingMovies        Contract  \
0               No          No          No              No  Month-to-month   
1              Yes          No          No              No        One year   

  PaperlessBilling     PaymentMethod  MonthlyCharges  TotalCharges  Churn  
0              Yes  Electronic check           29.85         29.85      0  
1               No      Mailed check           56.95       1889.50      0  


/tmp/ipykernel_1401/3986002096.py:17: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)


# Step 3: Data Partitioning (Train/Test Stratified Split)
Is step mein hum apnay data ko do zaroori hisson mein taqseem (split) kar rahe hain:
1. `X` (Features/Inputs): Churn ke ilawa baqi tamam columns jo customer ki maloomat hain.
2. `y` (Target/Output): `Churn` column jo model ne predict karna seekhna hai.

Hum Python se automatic kehte hain ke numerical (numbers) aur categorical (text) columns ko alag kare. Phir `train_test_split` ke zariye 80% data model ko parhanay (Train) ke liye aur 20% data model ka test lenay (Test) ke liye alag kar dete hain. Yahan `stratify=y` lagana bohot zaroori hai taake train aur test dono hisson mein chor kar jaanay wale aur ruknay wale customers ki ratio/percentage barabar rahe.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["Churn"])
y = df["Churn"]

# Identify Numerical and Categorical columns automatically
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Numerical columns: {num_cols}")
print(f"Categorical columns: {cat_cols}")
print("Data successfully split into Train and Test sets!")

/tmp/ipykernel_1401/404683454.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=["object"]).columns.tolist()


Numerical columns: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
Categorical columns: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
Data successfully split into Train and Test sets!


# Step 4: Production-Ready Preprocessing Pipeline (ColumnTransformer)
Is step mein hum data preprocessing ki aisi automatic machine (Pipeline) tayyar kar rahe hain jo bagair kisi data leakage ke numerical aur categorical data ko aik sath transform karegi:

1. **Numerical Pipeline:** Jo columns numbers wale hain (`num_cols`), agar un mein koi value missing hogi toh yeh use `median` se fill karega (`SimpleImputer`) aur phir saare numbers ka scale barabar karne ke liye unhein normalize (`StandardScaler`) kar dega.
2. **Categorical Pipeline:** Jo columns text/categories wale hain (`cat_cols`), unki missing values ko sab se zyada baar aane wale word se fill karega (`most_frequent`) aur phir un text values ko machine ke samajhne ke liye $0$ aur $1$ ki binary matrix mein badal dega (`OneHotEncoder`).

Aakhiri step mein hum `ColumnTransformer` use kar ke in dono pipelines ko ek single unified block (`preprocessor`) mein fuse (jadd) kar dete hain taake train aur test data par bilkul ek jaise rules lagayein ja sakein.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# Numerical Pipeline (Imputation + Scaling)
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical Pipeline (Imputation + One-Hot Encoding)
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine transformers using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ]
)

print("Preprocessing Pipeline successfully created!")

Preprocessing Pipeline successfully created!


# Step 5: Model Grid Setup & Hyperparameter Tuning (GridSearchCV)
Is step mein hum do zaroori kaam aik sath kar rahe hain:

1. **Full Pipeline Connect Karna:** Humne pichlay step mein jo `preprocessor` banaya tha, usay hum **Random Forest Classifier** model ke sath connect kar ke aik `full_pipeline` bana rahe hain. Iska faida yeh hai ke raw data automatic pehle process hoga aur phir seedha model ke paas training ke liye chala jayega (zero manual effort).
2. **Hyperparameter Tuning (GridSearchCV):** Random Forest ke paas different options hoti hain jaise kitne trees banane hain (`n_estimators`) ya unki depth kitni rakhni hai (`max_depth`). Hum Python ko aik list (grid) de dete hain. `GridSearchCV` automatic 3-fold cross-validation chalata hai—yani data ke different hissay kar ke har parameter combination ko check karta hai—aur jo sab se best perform karta hai (F1-Score ke mutabiq), us best model ko final select kar leta hai.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

# Create the full training pipeline with Random Forest
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

# Define the grid of hyperparameters to search
param_grid = {
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [5, 10, None],
    'classifier__min_samples_split': [2, 5]
}

print("Starting GridSearchCV for Hyperparameter Tuning... (This may take a minute)")
grid_search = GridSearchCV(full_pipeline, param_grid, cv=3, scoring='f1', n_jobs=-1)
grid_search.fit(X_train, y_train)

print("GridSearchCV Complete!")
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best Train F1-Score: {grid_search.best_score_:.4f}")

Starting GridSearchCV for Hyperparameter Tuning... (This may take a minute)
GridSearchCV Complete!
Best Parameters: {'classifier__max_depth': 10, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 100}
Best Train F1-Score: 0.5804


# Step 6: Model Validation & Performance Evaluation
Is step mein hum apnay trained model ka unseen data (Test Set) par final test le rahe hain:

1. **Best Pipeline Utarna:** `grid_search.best_estimator_` ke zariye hum us poori pipeline ko nikalte hain jis ke paas best preprocessor aur best tuned parameters hain.
2. **Predictions Chalana:** Hum test data (`X_test`) model ko dete hain aur woh har customer ke liye predict karta hai ke yeh company chorega (1) ya rukega (0).
3. **Metrics Check Karna:** Hum teen tarah se model ki performance check karte hain:
   * **Accuracy Score:** Overall kitni predictions sahi thin.
   * **F1 Score (Macro):** Hamare data mein class imbalance hota hai (yani ruknay wale customers zyada hote hain aur chornay wale kam). F1-Score humein yeh confirm karta hai ke model minority class (Churn=1) ko bhi sahi tarike se pehchan pa raha hai ya nahi.
   * **Classification Report:** Yeh humein Precision aur Recall ki poori breakdown dikhati hai taake hum model ki kamzoriyan samajh sakein.

In [ ]:
from sklearn.metrics import classification_report, accuracy_score, f1_score

# Get the best estimator pipeline
best_pipeline = grid_search.best_estimator_

# Make predictions on test data
y_pred = best_pipeline.predict(X_test)

print("--- Test Set Evaluation ---")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1 Score (Macro): {f1_score(y_test, y_pred, average='macro'):.4f}\n")
print("Classification Report:\n", classification_report(y_test, y_pred))

--- Test Set Evaluation ---
Accuracy Score: 0.7991
F1 Score (Macro): 0.7238

Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.90      0.87      1035
           1       0.65      0.52      0.58       374

    accuracy                           0.80      1409
   macro avg       0.75      0.71      0.72      1409
weighted avg       0.79      0.80      0.79      1409



# Step 7: Pipeline Serialization (Joblib Export)
Is step mein hum apnay pooray engineering system ko freeze aur save kar rahe hain:

Hum **Joblib** library ka use karte hue `best_pipeline` (jis mein data saaf karne ka tarika, scale karne ka tarika, aur final trained Random Forest model sab kuch aik sath majood hai) ko ek single file `telco_churn_pipeline.pkl` mein convert kar ke computer ki hard disk par permanently save kar dete hain.

Is file ka sab se bada faida yeh hai ke jab hum isay deployment (Gradio app ya kisi server) mein use karenge, toh humein raw data ko alag se preprocess nahi karna padega. Yeh `.pkl` file naye data ko automatic handle kar ke direct prediction nikal degi.

In [ ]:
import joblib

# Exporting the production-ready pipeline
model_filename = "telco_churn_pipeline.pkl"
joblib.dump(best_pipeline, model_filename)

print(f"Production pipeline successfully exported as '{model_filename}'!")

Production pipeline successfully exported as 'telco_churn_pipeline.pkl'!


# Step 8: Application Deployment (Gradio Interactive Interface)
Is aakhiri step mein hum apnay trained model ko ek live interactive web app mein badal rahay hain:

1. **Saved Pipeline Load Karna:** Hum `joblib.load` ke zariye pichlay step mein save ki gayi `.pkl` file ko dobara dimaag mein load kartay hain.
2. **Prediction Function:** `predict_churn` ka function user se form ke zariye saari input fields leta hai, unhein ek single-row Pandas DataFrame mein convert karta hai, aur pipeline par chalata hai. Pipeline khud data ko saaf karti hai aur model se pooch kar batati hai ke customer ke chor kar jaanay ki probability (chances) kitnay percent hain.
3. **Gradio UI Form Design:** Hum dropdowns, radio buttons, sliders, aur numeric input boxes use kartay hain taake user ko manually mushkil codes na likhnay parein.

`interface.launch(share=True)` ko run kartay hi Colab ke andar ek pyara sa form chal paray ga aur sath hi ek public shareable link (`*.gradio.live`) bhi banay ga, jisay aap kisi ko bhi bhej kar apna live project check karwa sakti hain!

In [ ]:
import gradio as gr
import joblib
import pandas as pd

# 1. Load the saved pipeline
pipeline = joblib.load("telco_churn_pipeline.pkl")

# 2. Define prediction function for Gradio
def predict_churn(gender, SeniorCitizen, Partner, Dependents, tenure, PhoneService, MultipleLines,
                  InternetService, OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport,
                  StreamingTV, StreamingMovies, Contract, PaperlessBilling, PaymentMethod,
                  MonthlyCharges, TotalCharges):

    # Create a DataFrame with the exact column names required by the pipeline
    input_data = pd.DataFrame([{
        'gender': gender, 'SeniorCitizen': int(SeniorCitizen), 'Partner': Partner, 'Dependents': Dependents,
        'tenure': int(tenure), 'PhoneService': PhoneService, 'MultipleLines': MultipleLines,
        'InternetService': InternetService, 'OnlineSecurity': OnlineSecurity, 'OnlineBackup': OnlineBackup,
        'DeviceProtection': DeviceProtection, 'TechSupport': TechSupport, 'StreamingTV': StreamingTV,
        'StreamingMovies': StreamingMovies, 'Contract': Contract, 'PaperlessBilling': PaperlessBilling,
        'PaymentMethod': PaymentMethod, 'MonthlyCharges': float(MonthlyCharges), 'TotalCharges': float(TotalCharges)
    }])

    # Get probability and prediction
    prob = pipeline.predict_proba(input_data)[0][1] # Probability of Churning (Class 1)
    prediction = pipeline.predict(input_data)[0]

    # Format output
    status = "⚠️ High Risk of Churning!" if prediction == 1 else "✅ Safe! Customer is likely to Stay."
    return {"Result": status, "Churn Probability": f"{prob*100:.2f}%"}

# 3. Build Gradio UI with matching inputs
inputs = [
    gr.Dropdown(["Male", "Female"], label="Gender"),
    gr.Radio(["0", "1"], label="Senior Citizen (1=Yes, 0=No)"),
    gr.Dropdown(["Yes", "No"], label="Has Partner?"),
    gr.Dropdown(["Yes", "No"], label="Has Dependents?"),
    gr.Slider(0, 72, step=1, value=12, label="Tenure (Months)"),
    gr.Dropdown(["Yes", "No"], label="Phone Service"),
    gr.Dropdown(["No phone service", "No", "Yes"], label="Multiple Lines"),
    gr.Dropdown(["DSL", "Fiber optic", "No"], label="Internet Service Type"),
    gr.Dropdown(["No", "Yes", "No internet service"], label="Online Security"),
    gr.Dropdown(["No", "Yes", "No internet service"], label="Online Backup"),
    gr.Dropdown(["No", "Yes", "No internet service"], label="Device Protection"),
    gr.Dropdown(["No", "Yes", "No internet service"], label="Tech Support"),
    gr.Dropdown(["No", "Yes", "No internet service"], label="Streaming TV"),
    gr.Dropdown(["No", "Yes", "No internet service"], label="Streaming Movies"),
    gr.Dropdown(["Month-to-month", "One year", "Two year"], label="Contract Type"),
    gr.Dropdown(["Yes", "No"], label="Paperless Billing"),
    gr.Dropdown(["Electronic check", "Mailed check", "Bank transfer (automatic)", "Credit card (automatic)"], label="Payment Method"),
    gr.Number(value=50.0, label="Monthly Charges ($)"),
    gr.Number(value=500.0, label="Total Charges ($)")
]

interface = gr.Interface(
    fn=predict_churn,
    inputs=inputs,
    outputs="json",
    title="📊 Telco Customer Churn Predictor Pipeline",
    description="Enter customer details below to predict if they will leave (Churn) or stay using an end-to-end tuned Scikit-Learn Pipeline.",
    theme="teal"
)

# Launch with share=True to get a public link
interface.launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  super().__init__(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/gradio/utils.py:583: UserWarning: Cannot load teal. Caught Exception: Client error '404 Not Found' for url 'https://huggingface.co/api/spaces/teal' (Request ID: Root=1-6a25215d-2f4a779c21fc

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://653494a56adb233ede.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 📊 Final Summary & Key Insights

### 🛠️ Pipeline Architecture Overview

### 🔑 Key Takeaways

| Aspect | Detail |
|--------|--------|
| **Dataset** | IBM Telco Customer Churn Dataset |
| **Problem Type** | Binary Classification (0 = Stay, 1 = Churn) |
| **Data Leakage Fix** | Strict use of Scikit-learn Pipeline API for zero train-test contamination |
| **Optimization** | GridSearchCV with 3-Fold Stratified Cross-Validation |
| **Core Metric** | F1-Score Macro (due to high class imbalance in churn data) |
| **Artifact Export** | Joblib Serialization into a single reusable `telco_churn_pipeline.pkl` file |

---

### 📝 Observations

1. **Automation Over Manual Coding** — `ColumnTransformer` aur `Pipeline` ka use karne se hamara data preprocessing aur model prediction aik single command chain mein bind ho gaya hai, jo code ko production-ready banata hai.

2. **Handling Class Imbalance** — Dataset mein chor kar jaane wale (Churn) customers ki kitaad rukne wale customers se bohot kam thi. Training mein `stratify=y` aur tuning mein `scoring='f1'` use karne se model minority class ko bias-free pehchanna seekh gaya.

3. **No More Data Leakage** — Preprocessing steps (Imputation aur Scaling) ko pipeline ke andar rakhne se test data ki information training ke dauran kabhi leak nahi hui, jis se real-world testing accurate milti hai.

4. **Hyperparameter Selection** — `GridSearchCV` ne automatic check karke best performing tree depth (`max_depth`) aur ensemble size (`n_estimators`) chun li, jis se model over-fitting se bach gaya.

5. **Deployment Readiness** — Gradio UI direct exported `.pkl` file ko load karti hai. Iska matlab yeh hai ke live web application ko github ya server par jab koi raw input milega, toh use pipeline khud back-end par saaf aur scale karegi.

---

### 🚀 Possible Improvements
- Handle class imbalance strictly using advanced techniques like **SMOTE** (Synthetic Minority Over-sampling Technique).
- Try boosting algorithms like **XGBoost**, **LightGBM**, or **CatBoost** to challenge the Random Forest baseline.
- Add Feature Importance analysis to extract and plot which customer attributes (like contract type or monthly charges) contribute the most to churn behavior.
- Implement threshold tuning on `predict_proba` to catch high-risk churners more aggressively according to business requirements.

---
*Task 2 — DeveloperHub Corporation AI/ML Internship*